In [18]:
!wget --no-check-certificate 'https://raw.githubusercontent.com/nursace/ai-framework-slides/main/data/campus-life-gpa.csv' -O campus-life-gpa.csv


--2026-10-22 13:14:55--  https://raw.githubusercontent.com/nursace/ai-framework-slides/main/data/campus-life-gpa.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 299585 (293K) [text/plain]
Saving to: ‘campus-life-gpa.csv’

campus-life-gpa.csv 100%[===================>] 292.56K  --.-KB/s    in 0.03s   

2026-10-22 13:14:55 (10.4 MB/s) - ‘campus-life-gpa.csv’ saved [299585/299585]



In [20]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv('campus-life-gpa.csv')

print(df.shape)
display(df.head())

feature_cols = [
    'sleep_hours_avg',
    'study_hours_week',
    'library_minutes_week',
    'phone_screen_hours_day',
    'cafe_visits_week',
    'coffee_cups_week',
    'stress_1_10',
    'steps_avg_day',
    'commute_minutes',
    'part_time_hours_week',
    'group_study_sessions',
    'student_year'
]

X = pd.get_dummies(
    df[feature_cols + ['faculty']],
    columns=['faculty'],
    dtype=float
)

y = df['gpa'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)
print('обучающая выборка:', X_train.shape)
print('тестовая выборка:', X_test.shape)

(5000, 15)


,student_id,faculty,student_year,sleep_hours_avg,study_hours_week,library_minutes_week,phone_screen_hours_day,cafe_visits_week,coffee_cups_week,stress_1_10,steps_avg_day,commute_minutes,part_time_hours_week,group_study_sessions,gpa
0,S00001,Arts,2,6.8,14.4,154.0,6.7,2,6,8,7852,64,12.1,2,2.56
1,S00002,CS,1,5.5,13.8,200.0,5.4,4,9,7,3147,44,17.7,1,2.49
2,S00003,Business,3,6.6,14.8,195.0,5.9,5,7,8,7962,30,0.0,3,2.39
3,S00004,Arts,3,5.8,10.7,217.0,4.2,3,7,6,8562,22,0.0,0,3.25
4,S00005,Engineering,2,6.5,24.6,339.0,3.8,4,7,5,6575,30,2.2,3,3.65


обучающая выборка: (3750, 17)
тестовая выборка: (1250, 17)


1

In [21]:
lr_plain = LinearRegression()
lr_plain.fit(X_train, y_train)
pred_plain = lr_plain.predict(X_test)

lr_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

lr_scaled.fit(X_train, y_train)
pred_scaled = lr_scaled.predict(X_test)

r2_plain = r2_score(y_test, pred_plain)
r2_scaled = r2_score(y_test, pred_scaled)

print(f'R^2 без масштабирования: {r2_plain:.6f}')
print(f'R^2 с масштабированием:  {r2_scaled:.6f}')
print(f'разница: {r2_scaled - r2_plain:.6f}')

R^2 без масштабирования: 0.682983
R^2 с масштабированием:  0.682983
разница: -0.000000


2

In [22]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold

alphas = [0.1, 1, 10, 50, 100, 500]

ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    ridge_pipeline,
    param_grid={'model__alpha': alphas},
    scoring='r2',
    cv=cv
)

search.fit(X_train, y_train)

alpha_results = pd.DataFrame({
    'alpha': [
        p['model__alpha']
        for p in search.cv_results_['params']
    ],
    'средний R^2 на CV': search.cv_results_['mean_test_score']
})

print(alpha_results.round(6).to_string(index=False))
print('лучший alpha:', search.best_params_['model__alpha'])

best_ridge = search.best_estimator_
pred_ridge = best_ridge.predict(X_test)

comparison = pd.DataFrame({
    'модель': ['LinearRegression', 'Ridge'],
    'Test R^2': [
        r2_score(y_test, pred_scaled),
        r2_score(y_test, pred_ridge)
    ],
    'Test MSE': [
        mean_squared_error(y_test, pred_scaled),
        mean_squared_error(y_test, pred_ridge)
    ],
    'max_|coef|': [
        abs(lr_scaled.named_steps['model'].coef_).max(),
        abs(best_ridge.named_steps['model'].coef_).max()
    ]
})

print('сравнение моделей:')
print(comparison.round(6).to_string(index=False))

 alpha  средний R^2 на CV
   0.1           0.664573
   1.0           0.664575
  10.0           0.664576
  50.0           0.664354
 100.0           0.663670
 500.0           0.651290
лучший alpha: 10
сравнение моделей:
          модель  Test R^2  Test MSE  max_|coef|
LinearRegression  0.682983  0.072599    0.275791
           Ridge  0.683047  0.072584    0.273564


3

In [23]:
two_features = ['study_hours_week', 'sleep_hours_avg']

lr_two = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

lr_two.fit(X_train[two_features], y_train)
pred_two = lr_two.predict(X_test[two_features])

r2_two = r2_score(y_test, pred_two)
r2_full = r2_score(y_test, pred_scaled)

print(f'Test R^2 полной модели: {r2_full:.6f}')
print(f'Test R^2 модели с двумя признаками: {r2_two:.6f}')

difference = r2_full - r2_two

if difference > 0:
    print(f'модель с двумя признаками хуже на {difference:.6f} по R^2.')
elif difference < 0:
    print(f'модель с двумя признаками лучше на {-difference:.6f} по R^2.')
else:
    print('качество моделей одинаковое.')

Test R^2 полной модели: 0.682983
Test R^2 модели с двумя признаками: 0.471537
модель с двумя признаками хуже на 0.211447 по R^2.
